# 05 - LightGBM global model v1

## 1. Load data

In [14]:
import pandas as pd
import numpy as np
from pathlib import Path

import lightgbm as lgb
from lightgbm import LGBMRegressor

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)

ROOT = Path("..")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs"

SUB_DIR = OUTPUTS / "submissions"
MODEL_DIR = OUTPUTS / "models"
CV_DIR = OUTPUTS / "cv_results"

SUB_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
CV_DIR.mkdir(parents=True, exist_ok=True)

daily_panel = pd.read_parquet(DATA_PROCESSED / "daily_panel.parquet")
sku_activity = pd.read_parquet(DATA_PROCESSED / "sku_activity.parquet")
sample = pd.read_csv(DATA_RAW / "sample_submission.csv")

daily_panel["Date"] = pd.to_datetime(daily_panel["Date"])

print("daily_panel:", daily_panel.shape)
print("sku_activity:", sku_activity.shape)
print("sample:", sample.shape)

daily_panel: (28014888, 17)
sku_activity: (15972, 20)
sample: (31944, 29)


## 2. Config

In [15]:
FINAL_TRAIN_END = pd.Timestamp("2025-09-05")
FINAL_FORECAST_START = pd.Timestamp("2025-09-06")
HORIZON = 56

TARGET_COL = "y_net_clip"  
RANDOM_STATE = 42

## 3. Create SKU x Date matrix

In [16]:
panel_small = daily_panel[["ItemCode", "Date", TARGET_COL]].copy()

y_wide = (
    panel_small
    .pivot(index="ItemCode", columns="Date", values=TARGET_COL)
    .sort_index(axis=0)
    .sort_index(axis=1)
)

itemcodes = y_wide.index.to_numpy()
dates = pd.to_datetime(y_wide.columns)
date_to_pos = {pd.Timestamp(d): i for i, d in enumerate(dates)}

y_mat = y_wide.to_numpy(dtype=np.float32)

print("y_mat shape:", y_mat.shape)
print("date range:", dates[0], "→", dates[-1])
print("n itemcodes:", len(itemcodes))

y_mat shape: (15972, 1754)
date range: 2020-11-17 00:00:00 → 2025-09-05 00:00:00
n itemcodes: 15972


## 4. Create metadata for SKU

In [17]:
meta = sku_activity.set_index("ItemCode").reindex(itemcodes).copy()

for col in ["positive_profit", "profit_rank", "active_days", "days_since_last_sale", "return_rate_qty"]:
    if col not in meta.columns:
        raise ValueError(f"Missing column in sku_activity: {col}")

meta["positive_profit"] = meta["positive_profit"].fillna(0)
meta["profit_rank"] = meta["profit_rank"].fillna(999999)
meta["active_days"] = meta["active_days"].fillna(0)
meta["days_since_last_sale"] = meta["days_since_last_sale"].fillna(9999)
meta["return_rate_qty"] = meta["return_rate_qty"].fillna(0)

metric_weight_arr = meta["positive_profit"].to_numpy(dtype=np.float32)
metric_weight_arr = metric_weight_arr / metric_weight_arr.sum()

profit_rank_arr = meta["profit_rank"].to_numpy(dtype=np.float32)
active_days_arr = meta["active_days"].to_numpy(dtype=np.float32)
return_rate_arr = meta["return_rate_qty"].to_numpy(dtype=np.float32)
positive_profit_arr = meta["positive_profit"].to_numpy(dtype=np.float32)

item_to_idx = {sku: i for i, sku in enumerate(itemcodes)}

display(meta.head())

,active_days,active_net_days,total_y_net,total_y_gross,total_return,total_sales,total_cost,total_profit,total_transactions,first_sale_date,last_sale_date,first_transaction_date,last_transaction_date,has_ever_sold,days_since_last_sale,days_since_last_transaction,positive_profit,profit_rank,return_rate_qty
ItemCode,,,,,,,,,,,,,,,,,,,
SKU-00001,15,15,30.0,30.0,0.0,3.608433e+07,0.0,3.608433e+07,30.0,2025-05-26,2025-08-28,2025-05-26,2025-08-28,1,8,8,3.608433e+07,781,0.0
SKU-00002,895,895,5894.0,5894.0,0.0,8.012686e+09,0.0,8.012686e+09,5896.0,2022-01-10,2025-09-05,2022-01-10,2025-09-05,1,0,0,8.012686e+09,2,0.0
SKU-00003,1061,1061,10935.0,10935.0,0.0,1.670068e+10,0.0,1.670068e+10,10936.0,2022-01-03,2025-09-04,2022-01-03,2025-09-04,1,1,1,1.670068e+10,1,0.0
SKU-00004,279,279,659.0,659.0,0.0,8.359968e+08,0.0,8.359968e+08,659.0,2023-07-19,2024-12-20,2023-07-19,2024-12-20,1,259,259,8.359968e+08,12,0.0
SKU-00005,327,327,1101.0,1101.0,0.0,2.243340e+09,0.0,2.243340e+09,1103.0,2022-01-03,2023-06-26,2022-01-03,2023-06-26,1,802,802,2.243340e+09,4,0.0


## 5. Choose SKU for traning

In [18]:
N_TOP = 3000
N_TAIL_SAMPLE = 1000

top_skus = (
    meta.sort_values("profit_rank")
    .head(N_TOP)
    .index
    .to_numpy()
)

tail_pool = meta.loc[~meta.index.isin(top_skus)].index.to_numpy()

rng = np.random.default_rng(RANDOM_STATE)
tail_sample = rng.choice(
    tail_pool,
    size=min(N_TAIL_SAMPLE, len(tail_pool)),
    replace=False
)

train_skus = np.unique(np.concatenate([top_skus, tail_sample]))
train_sku_idx = np.array([item_to_idx[x] for x in train_skus], dtype=int)

all_sku_idx = np.arange(len(itemcodes), dtype=int)

print("Train SKUs:", len(train_skus))
print("Top SKU included:", len(top_skus))
print("Tail sample:", len(tail_sample))

Train SKUs: 4000
Top SKU included: 3000
Tail sample: 1000


## 6. Choose cutoff dates to create training data

In [19]:
max_cutoff = FINAL_TRAIN_END - pd.Timedelta(days=HORIZON)

cutoff_dates = pd.date_range(
    start="2023-01-01",
    end=max_cutoff,
    freq="28D"
)

cutoff_dates = [pd.Timestamp(d) for d in cutoff_dates if pd.Timestamp(d) in date_to_pos]

print("Number of cutoff dates:", len(cutoff_dates))
print("First cutoff:", cutoff_dates[0])
print("Last cutoff:", cutoff_dates[-1])

Number of cutoff dates: 33
First cutoff: 2023-01-01 00:00:00
Last cutoff: 2025-06-15 00:00:00


## 6. Create features function

In [20]:
dow_arr = np.array([pd.Timestamp(d).dayofweek for d in dates])
month_arr = np.array([pd.Timestamp(d).month for d in dates])
day_arr = np.array([pd.Timestamp(d).day for d in dates])


def _safe_value(mat, sku_indices, pos):
    if pos < 0:
        return np.zeros(len(sku_indices), dtype=np.float32)
    return mat[sku_indices, pos].astype(np.float32)


def _rolling_stats(mat, sku_indices, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    vals = mat[np.ix_(sku_indices, np.arange(start_pos, end_pos + 1))]
    
    mean = vals.mean(axis=1).astype(np.float32)
    std = vals.std(axis=1).astype(np.float32)
    maxv = vals.max(axis=1).astype(np.float32)
    active = (vals > 0).sum(axis=1).astype(np.float32)
    
    return mean, std, maxv, active


def _same_dow_mean(mat, sku_indices, cutoff_pos, target_dow, n_last=13):
    hist_positions = np.where(dow_arr[:cutoff_pos + 1] == target_dow)[0]
    hist_positions = hist_positions[-n_last:]
    
    if len(hist_positions) == 0:
        return np.zeros(len(sku_indices), dtype=np.float32)
    
    vals = mat[np.ix_(sku_indices, hist_positions)]
    return vals.mean(axis=1).astype(np.float32)


def _days_since_last_sale(mat, sku_indices, cutoff_pos, lookback=365):
    start_pos = max(0, cutoff_pos - lookback + 1)
    vals = mat[np.ix_(sku_indices, np.arange(start_pos, cutoff_pos + 1))]
    positive = vals > 0
    
    rev = positive[:, ::-1]
    has_sale = rev.any(axis=1)
    dist = np.where(has_sale, rev.argmax(axis=1), 9999)
    
    return dist.astype(np.float32)


def build_feature_block(sku_indices, cutoff_pos, horizon):
    """
    Build features for a given cutoff position and forecast horizon.

    Important:
    - During training, target date exists inside historical dates.
    - During final prediction, target date is beyond historical dates.
    Therefore target_date must be computed from cutoff_date + horizon,
    not by indexing dates[cutoff_pos + horizon].
    """
    cutoff_date = pd.Timestamp(dates[cutoff_pos])
    target_date = cutoff_date + pd.Timedelta(days=int(horizon))

    n = len(sku_indices)
    target_dow = target_date.dayofweek
    
    data = {
        "horizon": np.full(n, horizon, dtype=np.float32),
        "target_dayofweek": np.full(n, target_dow, dtype=np.float32),
        "target_is_saturday": np.full(n, int(target_dow == 5), dtype=np.float32),
        "target_is_sunday": np.full(n, int(target_dow == 6), dtype=np.float32),
        "target_month": np.full(n, target_date.month, dtype=np.float32),
        "target_day": np.full(n, target_date.day, dtype=np.float32),
    }
    
    # Last known values at cutoff
    data["qty_last_day"] = _safe_value(y_mat, sku_indices, cutoff_pos)
    data["qty_7_days_ago"] = _safe_value(y_mat, sku_indices, cutoff_pos - 6)
    data["qty_14_days_ago"] = _safe_value(y_mat, sku_indices, cutoff_pos - 13)
    data["qty_28_days_ago"] = _safe_value(y_mat, sku_indices, cutoff_pos - 27)
    data["qty_56_days_ago"] = _safe_value(y_mat, sku_indices, cutoff_pos - 55)
    data["qty_364_days_ago"] = _safe_value(y_mat, sku_indices, cutoff_pos - 363)
    
    # Rolling stats
    for window in [7, 14, 28, 56, 112, 180]:
        mean, std, maxv, active = _rolling_stats(y_mat, sku_indices, cutoff_pos, window)
        data[f"roll_mean_{window}"] = mean
        data[f"roll_std_{window}"] = std
        data[f"roll_max_{window}"] = maxv
        data[f"active_days_{window}"] = active
    
    # Same day-of-week stats
    data["same_dow_mean_4"] = _same_dow_mean(y_mat, sku_indices, cutoff_pos, target_dow, n_last=4)
    data["same_dow_mean_8"] = _same_dow_mean(y_mat, sku_indices, cutoff_pos, target_dow, n_last=8)
    data["same_dow_mean_13"] = _same_dow_mean(y_mat, sku_indices, cutoff_pos, target_dow, n_last=13)
    
    # Recency
    data["days_since_last_sale_365"] = _days_since_last_sale(y_mat, sku_indices, cutoff_pos, lookback=365)
    
    # Static SKU features
    data["log_positive_profit"] = np.log1p(positive_profit_arr[sku_indices]).astype(np.float32)
    data["log_profit_rank"] = np.log1p(profit_rank_arr[sku_indices]).astype(np.float32)
    data["active_days_total"] = active_days_arr[sku_indices].astype(np.float32)
    data["return_rate_qty"] = return_rate_arr[sku_indices].astype(np.float32)
    data["metric_weight"] = metric_weight_arr[sku_indices].astype(np.float32)
    
    return pd.DataFrame(data)

## 8. Generate training data

In [21]:
chunks = []

for ci, cutoff_date in enumerate(cutoff_dates, start=1):
    cutoff_pos = date_to_pos[cutoff_date]
    print(f"[{ci}/{len(cutoff_dates)}] cutoff:", cutoff_date.date())
    
    for h in range(1, HORIZON + 1):
        target_pos = cutoff_pos + h
        
        if target_pos >= len(dates):
            continue
        
        target_date = pd.Timestamp(dates[target_pos])
        
        if target_date > FINAL_TRAIN_END:
            continue
        
        X_block = build_feature_block(train_sku_idx, cutoff_pos, h)
        y_block = y_mat[train_sku_idx, target_pos].astype(np.float32)
        
        # Weight để model chú ý SKU profit cao hơn
        row_weight = 1.0 + 1000.0 * metric_weight_arr[train_sku_idx]
        row_weight = row_weight.astype(np.float32)
        
        X_block["target"] = y_block
        X_block["row_weight"] = row_weight
        X_block["cutoff_date"] = cutoff_date
        
        chunks.append(X_block)

train_df = pd.concat(chunks, ignore_index=True)

print("train_df shape:", train_df.shape)
display(train_df.head())
display(train_df["target"].describe())

[1/33] cutoff: 2023-01-01
[2/33] cutoff: 2023-01-29
[3/33] cutoff: 2023-02-26
[4/33] cutoff: 2023-03-26
[5/33] cutoff: 2023-04-23
[6/33] cutoff: 2023-05-21
[7/33] cutoff: 2023-06-18
[8/33] cutoff: 2023-07-16
[9/33] cutoff: 2023-08-13
[10/33] cutoff: 2023-09-10
[11/33] cutoff: 2023-10-08
[12/33] cutoff: 2023-11-05
[13/33] cutoff: 2023-12-03
[14/33] cutoff: 2023-12-31
[15/33] cutoff: 2024-01-28
[16/33] cutoff: 2024-02-25
[17/33] cutoff: 2024-03-24
[18/33] cutoff: 2024-04-21
[19/33] cutoff: 2024-05-19
[20/33] cutoff: 2024-06-16
[21/33] cutoff: 2024-07-14
[22/33] cutoff: 2024-08-11
[23/33] cutoff: 2024-09-08
[24/33] cutoff: 2024-10-06
[25/33] cutoff: 2024-11-03
[26/33] cutoff: 2024-12-01
[27/33] cutoff: 2024-12-29
[28/33] cutoff: 2025-01-26
[29/33] cutoff: 2025-02-23
[30/33] cutoff: 2025-03-23
[31/33] cutoff: 2025-04-20
[32/33] cutoff: 2025-05-18
[33/33] cutoff: 2025-06-15
train_df shape: (7392000, 48)


,horizon,target_dayofweek,target_is_saturday,target_is_sunday,target_month,target_day,qty_last_day,qty_7_days_ago,qty_14_days_ago,qty_28_days_ago,qty_56_days_ago,qty_364_days_ago,roll_mean_7,roll_std_7,roll_max_7,active_days_7,roll_mean_14,roll_std_14,roll_max_14,active_days_14,roll_mean_28,roll_std_28,roll_max_28,active_days_28,roll_mean_56,roll_std_56,roll_max_56,active_days_56,roll_mean_112,roll_std_112,roll_max_112,active_days_112,roll_mean_180,roll_std_180,roll_max_180,active_days_180,same_dow_mean_4,same_dow_mean_8,same_dow_mean_13,days_since_last_sale_365,log_positive_profit,log_profit_rank,active_days_total,return_rate_qty,metric_weight,target,row_weight,cutoff_date
0,1.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.00,0.000,0.000000,9999.0,17.401369,6.661855,15.0,0.0,0.000210,0.0,1.209553,2023-01-01
1,1.0,0.0,0.0,0.0,1.0,2.0,0.0,9.0,13.0,4.0,4.0,0.0,9.285714,7.284313,18.0,5.0,7.928571,6.341071,18.0,11.0,6.071429,5.344748,18.0,23.0,4.803571,4.672945,18.0,46.0,4.401786,4.270873,24.0,90.0,3.311111,3.808308,24.0,130.0,9.25,7.375,6.538462,2.0,22.804293,1.098612,895.0,0.0,0.046532,3.0,47.532059,2023-01-01
2,1.0,0.0,0.0,0.0,1.0,2.0,0.0,18.0,14.0,7.0,5.0,2.0,11.142858,7.376189,18.0,5.0,11.642858,6.788721,18.0,11.0,9.285714,6.204014,18.0,23.0,8.107142,5.872693,21.0,47.0,7.017857,5.869081,34.0,95.0,5.744444,5.344906,34.0,148.0,11.75,11.125,9.230769,2.0,23.538715,0.693147,1061.0,0.0,0.096986,0.0,97.985832,2023-01-01
3,1.0,0.0,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.00,0.000,0.000000,9999.0,20.544136,2.564949,279.0,0.0,0.004855,0.0,5.854882,2023-01-01
4,1.0,0.0,0.0,0.0,1.0,2.0,0.0,3.0,4.0,1.0,2.0,7.0,1.000000,1.195229,3.0,3.0,1.428571,1.293627,4.0,9.0,1.142857,1.124858,4.0,18.0,1.696429,1.522681,6.0,41.0,2.080357,2.252532,14.0,79.0,2.800000,3.028384,20.0,134.0,2.25,2.500,2.384615,4.0,21.531233,1.609438,327.0,0.0,0.013028,0.0,14.027747,2023-01-01


count    7.392000e+06
mean     3.902343e-01
std      1.303843e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.035800e+04
Name: target, dtype: float64

## 8. Train/valid split

In [22]:
valid_cutoffs = sorted(cutoff_dates)[-2:]

is_valid = train_df["cutoff_date"].isin(valid_cutoffs)

drop_cols = ["target", "row_weight", "cutoff_date"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train = train_df.loc[~is_valid, feature_cols]
y_train = train_df.loc[~is_valid, "target"]
w_train = train_df.loc[~is_valid, "row_weight"]

X_valid = train_df.loc[is_valid, feature_cols]
y_valid = train_df.loc[is_valid, "target"]
w_valid = train_df.loc[is_valid, "row_weight"]

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("Number of features:", len(feature_cols))
print(feature_cols)

X_train: (6944000, 45)
X_valid: (448000, 45)
Number of features: 45
['horizon', 'target_dayofweek', 'target_is_saturday', 'target_is_sunday', 'target_month', 'target_day', 'qty_last_day', 'qty_7_days_ago', 'qty_14_days_ago', 'qty_28_days_ago', 'qty_56_days_ago', 'qty_364_days_ago', 'roll_mean_7', 'roll_std_7', 'roll_max_7', 'active_days_7', 'roll_mean_14', 'roll_std_14', 'roll_max_14', 'active_days_14', 'roll_mean_28', 'roll_std_28', 'roll_max_28', 'active_days_28', 'roll_mean_56', 'roll_std_56', 'roll_max_56', 'active_days_56', 'roll_mean_112', 'roll_std_112', 'roll_max_112', 'active_days_112', 'roll_mean_180', 'roll_std_180', 'roll_max_180', 'active_days_180', 'same_dow_mean_4', 'same_dow_mean_8', 'same_dow_mean_13', 'days_since_last_sale_365', 'log_positive_profit', 'log_profit_rank', 'active_days_total', 'return_rate_qty', 'metric_weight']


## 10. Train LightGBM

In [23]:
model = LGBMRegressor(
    objective="regression",
    n_estimators=1500,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    min_child_samples=100,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    sample_weight=w_train,
    eval_set=[(X_valid, y_valid)],
    eval_sample_weight=[w_valid],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=80),
        lgb.log_evaluation(period=100)
    ]
)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.439248 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 7445
[LightGBM] [Info] Number of data points in the train set: 6944000, number of used features: 44
[LightGBM] [Info] Start training from score 1.259922
Training until validation scores don't improve for 80 rounds
[100]	valid_0's rmse: 12.6621	valid_0's l2: 160.328
Early stopping, best iteration is:
[44]	valid_0's rmse: 12.4006	valid_0's l2: 153.776


,boosting_type,'gbdt'
,num_leaves,64
,max_depth,-1
,learning_rate,0.03
,n_estimators,1500
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,100


See feature importance

In [24]:
fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

display(fi.head(30))

fi.to_csv(CV_DIR / "step5_lgbm_v1_feature_importance.csv", index=False)

,feature,importance
5,target_day,327
1,target_dayofweek,261
4,target_month,169
0,horizon,165
38,same_dow_mean_13,151
32,roll_mean_180,118
35,active_days_180,117
36,same_dow_mean_4,105
28,roll_mean_112,98
24,roll_mean_56,95


## 11. Predict 56 days in future

In [25]:
final_cutoff_pos = date_to_pos[FINAL_TRAIN_END]

pred_cols = []

for h in range(1, HORIZON + 1):
    print("Predict horizon:", h)
    
    X_final_h = build_feature_block(all_sku_idx, final_cutoff_pos, h)
    pred_h = model.predict(X_final_h[feature_cols])
    pred_h = np.clip(pred_h, 0, None).astype(np.float32)
    
    pred_cols.append(pred_h)

lgbm_pred_arr = np.column_stack(pred_cols)

forecast_dates = pd.date_range(FINAL_FORECAST_START, periods=HORIZON, freq="D")

lgbm_pred_56 = pd.DataFrame(
    lgbm_pred_arr,
    index=itemcodes,
    columns=forecast_dates
)

print(lgbm_pred_56.shape)
display(lgbm_pred_56.head())

print("min:", lgbm_pred_56.min().min())
print("max:", lgbm_pred_56.max().max())
print("mean:", lgbm_pred_56.values.mean())
print("total forecast:", lgbm_pred_56.values.sum())

Predict horizon: 1
Predict horizon: 2
Predict horizon: 3
Predict horizon: 4
Predict horizon: 5
Predict horizon: 6
Predict horizon: 7
Predict horizon: 8
Predict horizon: 9
Predict horizon: 10
Predict horizon: 11
Predict horizon: 12
Predict horizon: 13
Predict horizon: 14
Predict horizon: 15
Predict horizon: 16
Predict horizon: 17
Predict horizon: 18
Predict horizon: 19
Predict horizon: 20
Predict horizon: 21
Predict horizon: 22
Predict horizon: 23
Predict horizon: 24
Predict horizon: 25
Predict horizon: 26
Predict horizon: 27
Predict horizon: 28
Predict horizon: 29
Predict horizon: 30
Predict horizon: 31
Predict horizon: 32
Predict horizon: 33
Predict horizon: 34
Predict horizon: 35
Predict horizon: 36
Predict horizon: 37
Predict horizon: 38
Predict horizon: 39
Predict horizon: 40
Predict horizon: 41
Predict horizon: 42
Predict horizon: 43
Predict horizon: 44
Predict horizon: 45
Predict horizon: 46
Predict horizon: 47
Predict horizon: 48
Predict horizon: 49
Predict horizon: 50
Predict h

,2025-09-06,2025-09-07,2025-09-08,2025-09-09,2025-09-10,2025-09-11,2025-09-12,2025-09-13,2025-09-14,2025-09-15,2025-09-16,2025-09-17,2025-09-18,2025-09-19,2025-09-20,2025-09-21,2025-09-22,2025-09-23,2025-09-24,2025-09-25,2025-09-26,2025-09-27,2025-09-28,2025-09-29,2025-09-30,2025-10-01,2025-10-02,2025-10-03,2025-10-04,2025-10-05,2025-10-06,2025-10-07,2025-10-08,2025-10-09,2025-10-10,2025-10-11,2025-10-12,2025-10-13,2025-10-14,2025-10-15,2025-10-16,2025-10-17,2025-10-18,2025-10-19,2025-10-20,2025-10-21,2025-10-22,2025-10-23,2025-10-24,2025-10-25,2025-10-26,2025-10-27,2025-10-28,2025-10-29,2025-10-30,2025-10-31
SKU-00001,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606
SKU-00002,4.292743,0.700469,8.583734,8.558983,8.763144,8.763144,8.763144,4.280612,0.700469,8.307783,8.081311,8.307783,8.307783,8.307783,4.280612,0.700469,8.307783,8.081311,8.307783,8.307783,8.474713,4.280612,0.700469,8.474713,8.248241,7.752327,7.752327,7.752327,4.280243,0.700469,7.752327,7.619795,7.752327,7.847608,7.931737,4.280243,0.700469,8.307783,8.081311,8.307783,8.307783,8.307783,4.280612,0.700469,8.307783,8.081311,8.307783,8.307783,8.307783,4.280612,0.700469,8.474713,8.248241,8.474713,8.474713,8.474713
SKU-00003,5.784775,0.813997,10.012414,10.107695,10.310130,10.310130,10.366091,6.028663,0.813997,10.857412,10.181988,10.181988,10.181988,10.181988,6.028663,0.813997,10.857412,10.181988,10.181988,10.181988,10.181988,6.028663,0.813997,10.857412,10.181988,9.181006,9.181006,9.181006,5.784775,0.813997,9.181006,9.181006,9.181006,9.276288,9.478723,5.784775,0.813997,10.857412,10.181988,10.181988,10.181988,10.181988,6.028663,0.813997,10.857412,10.181988,10.181988,10.181988,10.181988,6.028663,0.813997,10.857412,10.181988,10.181988,10.181988,10.181988
SKU-00004,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985
SKU-00005,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985


min: 0.0
max: 361.8379
mean: 0.48005688
total forecast: 429378.25


## 12. Postprocess LightGBM prediction

In [26]:
def postprocess_pred_matrix(pred_56, sku_activity):
    pred = pred_56.copy()
    pred = pred.fillna(0).clip(lower=0)
    
    # Sunday zero
    sunday_cols = [c for c in pred.columns if pd.Timestamp(c).dayofweek == 6]
    pred.loc[:, sunday_cols] = 0.0
    
    meta_local = sku_activity.set_index("ItemCode").reindex(pred.index)
    
    days_since = meta_local["days_since_last_sale"].fillna(9999)
    active_days = meta_local["active_days"].fillna(0)
    profit_rank = meta_local["profit_rank"].fillna(999999)
    
    # Conservative inactive rule
    zero_mask = (
        ((days_since > 365) & (profit_rank > 500)) |
        ((days_since > 180) & (active_days <= 3) & (profit_rank > 100)) |
        ((days_since > 90) & (active_days <= 1) & (profit_rank > 100))
    )
    
    pred.loc[zero_mask, :] = 0.0
    
    return pred


lgbm_pred_56_pp = postprocess_pred_matrix(lgbm_pred_56, sku_activity)

print("After postprocess")
print("min:", lgbm_pred_56_pp.min().min())
print("max:", lgbm_pred_56_pp.max().max())
print("mean:", lgbm_pred_56_pp.values.mean())
print("total forecast:", lgbm_pred_56_pp.values.sum())

sunday_cols = [c for c in lgbm_pred_56_pp.columns if c.dayofweek == 6]
print("Sunday total:", lgbm_pred_56_pp[sunday_cols].sum().sum())

After postprocess
min: 0.0
max: 361.8379
mean: 0.24992938
total forecast: 223544.84
Sunday total: 0.0


## 13. Convert submission function

In [27]:
def make_kaggle_submission(
    pred_56: pd.DataFrame,
    sample: pd.DataFrame,
    output_path=None
) -> pd.DataFrame:
    f_cols = [f"F{i}" for i in range(1, 29)]

    if pred_56.shape[1] != 56:
        raise ValueError(f"pred_56 must have 56 forecast columns, got {pred_56.shape[1]}")

    pred = pred_56.copy()
    pred = pred.sort_index()
    pred = pred.fillna(0).clip(lower=0).astype(float)

    validation_part = pred.iloc[:, :28].copy()
    validation_part.columns = f_cols
    validation_part.index = validation_part.index + "_validation"

    evaluation_part = pred.iloc[:, 28:56].copy()
    evaluation_part.columns = f_cols
    evaluation_part.index = evaluation_part.index + "_evaluation"

    pred_sub = (
        pd.concat([validation_part, evaluation_part], axis=0)
        .reset_index()
        .rename(columns={"index": "id"})
    )

    sub = sample[["id"]].merge(
        pred_sub,
        on="id",
        how="left",
        validate="one_to_one"
    )

    assert sub.shape == sample.shape
    assert sub["id"].tolist() == sample["id"].tolist()
    assert sub["id"].nunique() == len(sub)

    values = sub[f_cols].to_numpy(dtype=float)

    if np.isnan(values).any():
        raise ValueError("Submission contains NaN.")

    if np.isinf(values).any():
        raise ValueError("Submission contains inf.")

    if (values < 0).any():
        raise ValueError("Submission contains negative values.")

    if output_path is not None:
        sub.to_csv(output_path, index=False)
        print("Saved:", output_path)

    return sub

## 14. Create pure LGBM submission

In [28]:
submission_lgbm_path = SUB_DIR / "submission_lgbm_v1.csv"

submission_lgbm_v1 = make_kaggle_submission(
    lgbm_pred_56_pp,
    sample=sample,
    output_path=submission_lgbm_path
)

display(submission_lgbm_v1.head())
print(submission_lgbm_v1.shape)
print("Total:", submission_lgbm_v1[[f"F{i}" for i in range(1, 29)]].sum().sum())
print("Max:", submission_lgbm_v1[[f"F{i}" for i in range(1, 29)]].max().max())

Saved: ../outputs/submissions/submission_lgbm_v1.csv


,id,F1,F2,F3,F4,F5,F6,F7,F8,F9,F10,F11,F12,F13,F14,F15,F16,F17,F18,F19,F20,F21,F22,F23,F24,F25,F26,F27,F28
0,SKU-00001_validation,0.430606,0.0,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.0,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.0,0.430606,0.430606,0.430606,0.430606,0.430606,0.430606,0.0,0.430606,0.430606,0.430606,0.430606,0.430606
1,SKU-00002_validation,4.292743,0.0,8.583734,8.558983,8.763144,8.763144,8.763144,4.280612,0.0,8.307783,8.081311,8.307783,8.307783,8.307783,4.280612,0.0,8.307783,8.081311,8.307783,8.307783,8.474713,4.280612,0.0,8.474713,8.248241,7.752327,7.752327,7.752327
2,SKU-00003_validation,5.784775,0.0,10.012414,10.107695,10.310130,10.310130,10.366091,6.028663,0.0,10.857412,10.181988,10.181988,10.181988,10.181988,6.028663,0.0,10.857412,10.181988,10.181988,10.181988,10.181988,6.028663,0.0,10.857412,10.181988,9.181006,9.181006,9.181006
3,SKU-00004_validation,0.477985,0.0,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.0,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.0,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.0,0.477985,0.477985,0.477985,0.477985,0.477985
4,SKU-00005_validation,0.477985,0.0,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.0,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.0,0.477985,0.477985,0.477985,0.477985,0.477985,0.477985,0.0,0.477985,0.477985,0.477985,0.477985,0.477985


(31944, 29)
Total: 223544.8402661085
Max: 361.837890625


## 15. Blend with baseline_v1

In [29]:
def submission_to_pred56(submission: pd.DataFrame, forecast_start) -> pd.DataFrame:
    f_cols = [f"F{i}" for i in range(1, 29)]
    forecast_dates = pd.date_range(forecast_start, periods=56, freq="D")
    
    sub = submission.copy()
    parsed = sub["id"].str.rsplit("_", n=1, expand=True)
    sub["ItemCode"] = parsed[0]
    sub["window"] = parsed[1]
    
    val = (
        sub[sub["window"] == "validation"]
        .set_index("ItemCode")[f_cols]
        .sort_index()
    )
    eva = (
        sub[sub["window"] == "evaluation"]
        .set_index("ItemCode")[f_cols]
        .sort_index()
    )
    
    pred56 = pd.concat([val, eva], axis=1)
    pred56.columns = forecast_dates
    pred56 = pred56.astype(float)
    
    return pred56


baseline_path = SUB_DIR / "submission_baseline_v1.csv"
baseline_sub = pd.read_csv(baseline_path)

baseline_pred_56 = submission_to_pred56(
    baseline_sub,
    forecast_start=FINAL_FORECAST_START
)

baseline_pred_56 = baseline_pred_56.reindex(index=lgbm_pred_56_pp.index).fillna(0)

print("baseline total:", baseline_pred_56.values.sum())
print("lgbm total:", lgbm_pred_56_pp.values.sum())

baseline total: 76100.31872252747
lgbm total: 223544.84


In [30]:
blend_configs = {
    "blend_lgbm60_base40": (0.60, 0.40),
    "blend_lgbm50_base50": (0.50, 0.50),
    "blend_lgbm40_base60": (0.40, 0.60),
}

blend_predictions = {}

for name, (w_lgbm, w_base) in blend_configs.items():
    pred_blend = (
        w_lgbm * lgbm_pred_56_pp +
        w_base * baseline_pred_56
    )
    
    pred_blend = postprocess_pred_matrix(pred_blend, sku_activity)
    blend_predictions[name] = pred_blend
    
    print("=" * 60)
    print(name)
    print("total:", pred_blend.values.sum())
    print("max:", pred_blend.max().max())
    print("mean:", pred_blend.values.mean())

blend_lgbm60_base40
total: 164567.0331424281
max: 521.3785779989055
mean: 0.18399054723268857
blend_lgbm50_base50
total: 149822.579494318
max: 579.2557771250965
mean: 0.16750583554067608
blend_lgbm40_base60
total: 135078.1277607985
max: 637.1329915100769
mean: 0.15102112598922948


In [33]:
def make_segmented_lgbm_blend(
    lgbm_pred_56: pd.DataFrame,
    baseline_pred_56: pd.DataFrame,
    sku_activity: pd.DataFrame,
    version: str = "v1"
) -> pd.DataFrame:
    """
    Segment-wise blend:
    - use more LGBM for high-profit SKUs
    - use baseline for long-tail / inactive SKUs
    """
    lgbm = lgbm_pred_56.copy().reindex(index=baseline_pred_56.index).fillna(0)
    base = baseline_pred_56.copy().reindex(index=lgbm.index).fillna(0)

    meta = sku_activity.set_index("ItemCode").reindex(lgbm.index)

    profit_rank = meta["profit_rank"].fillna(999999)
    days_since = meta["days_since_last_sale"].fillna(9999)
    active_days = meta["active_days"].fillna(0)

    w_lgbm = pd.Series(0.0, index=lgbm.index)

    if version == "v1_conservative":
        # Safe version
        w_lgbm.loc[profit_rank <= 100] = 0.45
        w_lgbm.loc[(profit_rank > 100) & (profit_rank <= 500)] = 0.30
        w_lgbm.loc[(profit_rank > 500) & (profit_rank <= 1000)] = 0.15
        w_lgbm.loc[(profit_rank > 1000) & (profit_rank <= 2000)] = 0.05
        w_lgbm.loc[profit_rank > 2000] = 0.00

    elif version == "v2_balanced":
        # Slightly more aggressive
        w_lgbm.loc[profit_rank <= 100] = 0.60
        w_lgbm.loc[(profit_rank > 100) & (profit_rank <= 500)] = 0.40
        w_lgbm.loc[(profit_rank > 500) & (profit_rank <= 1000)] = 0.20
        w_lgbm.loc[(profit_rank > 1000) & (profit_rank <= 2000)] = 0.08
        w_lgbm.loc[profit_rank > 2000] = 0.00

    else:
        raise ValueError("version must be v1_conservative or v2_balanced")

    # Reduce LGBM usage for inactive/sparse SKUs
    inactive_mask = (
        ((days_since > 180) & (profit_rank > 500)) |
        ((active_days <= 3) & (profit_rank > 500)) |
        ((days_since > 90) & (active_days <= 1) & (profit_rank > 100))
    )

    w_lgbm.loc[inactive_mask] = 0.0

    # Convert SKU-level weights to matrix blend
    w = w_lgbm.to_numpy()[:, None]

    pred = w * lgbm.to_numpy() + (1 - w) * base.to_numpy()

    pred = pd.DataFrame(
        pred,
        index=lgbm.index,
        columns=lgbm.columns
    )

    pred = postprocess_pred_matrix(pred, sku_activity)

    return pred

In [34]:
segblend_v1 = make_segmented_lgbm_blend(
    lgbm_pred_56_pp,
    baseline_pred_56,
    sku_activity,
    version="v1_conservative"
)

segblend_v2 = make_segmented_lgbm_blend(
    lgbm_pred_56_pp,
    baseline_pred_56,
    sku_activity,
    version="v2_balanced"
)

for name, pred in {
    "segblend_v1_conservative": segblend_v1,
    "segblend_v2_balanced": segblend_v2
}.items():
    print("=" * 60)
    print(name)
    print("total:", pred.values.sum())
    print("max:", pred.max().max())
    print("mean:", pred.values.mean())

    sunday_cols = [c for c in pred.columns if pd.Timestamp(c).dayofweek == 6]
    print("sunday total:", pred[sunday_cols].sum().sum())

segblend_v1_conservative
total: 76494.7305742962
max: 608.1943827917079
mean: 0.08552324891584402
sunday total: 0.0
segblend_v2_balanced
total: 76765.75450206113
max: 521.3785657918742
mean: 0.08582626124966586
sunday total: 0.0


In [31]:
for name, pred in blend_predictions.items():
    output_path = SUB_DIR / f"submission_{name}.csv"
    sub = make_kaggle_submission(
        pred,
        sample=sample,
        output_path=output_path
    )
    
    f_cols = [f"F{i}" for i in range(1, 29)]
    print(name, "shape:", sub.shape, "total:", sub[f_cols].sum().sum(), "max:", sub[f_cols].max().max())

Saved: ../outputs/submissions/submission_blend_lgbm60_base40.csv
blend_lgbm60_base40 shape: (31944, 29) total: 164567.03314242806 max: 521.3785779989055
Saved: ../outputs/submissions/submission_blend_lgbm50_base50.csv
blend_lgbm50_base50 shape: (31944, 29) total: 149822.57949431796 max: 579.2557771250965
Saved: ../outputs/submissions/submission_blend_lgbm40_base60.csv
blend_lgbm40_base60 shape: (31944, 29) total: 135078.1277607985 max: 637.1329915100769


In [35]:
segmented_predictions = {
    "segblend_v1_conservative": segblend_v1,
    "segblend_v2_balanced": segblend_v2
}

for name, pred in segmented_predictions.items():
    output_path = SUB_DIR / f"submission_{name}.csv"

    sub = make_kaggle_submission(
        pred,
        sample=sample,
        output_path=output_path
    )

    f_cols = [f"F{i}" for i in range(1, 29)]

    print("=" * 60)
    print(name)
    print("shape:", sub.shape)
    print("total:", sub[f_cols].sum().sum())
    print("max:", sub[f_cols].max().max())
    print("missing:", sub[f_cols].isna().sum().sum())
    print("negative:", (sub[f_cols] < 0).sum().sum())

Saved: ../outputs/submissions/submission_segblend_v1_conservative.csv
segblend_v1_conservative
shape: (31944, 29)
total: 76494.73057429622
max: 608.1943827917079
missing: 0
negative: 0
Saved: ../outputs/submissions/submission_segblend_v2_balanced.csv
segblend_v2_balanced
shape: (31944, 29)
total: 76765.75450206113
max: 521.3785657918742
missing: 0
negative: 0


## 16. Submission QA

In [32]:
candidate_path = SUB_DIR / "submission_blend_lgbm50_base50.csv"
candidate = pd.read_csv(candidate_path)

f_cols = [f"F{i}" for i in range(1, 29)]

print("Shape:", candidate.shape)
print("Unique IDs:", candidate["id"].nunique())
print("Missing:", candidate[f_cols].isna().sum().sum())
print("Negative:", (candidate[f_cols] < 0).sum().sum())
print("Total:", candidate[f_cols].sum().sum())
print("Max:", candidate[f_cols].max().max())

assert candidate.shape == sample.shape
assert candidate["id"].tolist() == sample["id"].tolist()
assert candidate[f_cols].isna().sum().sum() == 0
assert (candidate[f_cols] < 0).sum().sum() == 0

validation_rows = candidate["id"].str.endswith("_validation")
evaluation_rows = candidate["id"].str.endswith("_evaluation")

sunday_f = ["F2", "F9", "F16", "F23"]

print("Validation Sunday total:", candidate.loc[validation_rows, sunday_f].sum().sum())
print("Evaluation Sunday total:", candidate.loc[evaluation_rows, sunday_f].sum().sum())

Shape: (31944, 29)
Unique IDs: 31944
Missing: 0
Negative: 0
Total: 149822.57949431796
Max: 579.2557771250965
Validation Sunday total: 0.0
Evaluation Sunday total: 0.0
